# LunarLander LTLf training

This self-contained Kaggle notebook installs the dependencies, reconstructs every project source file, runs the LTLf-guided DDQN training, displays the generated plots, and packages the outputs. No Dataset or additional upload is required.

Enable a GPU from **Settings → Accelerator → GPU** before starting.

## 1. Install system and Python dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq mona graphviz swig
%pip install -q "gymnasium[box2d]" ltlf2dfa graphviz pandas matplotlib


## 2. Create the writable project directory

In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("/kaggle/working/lunarLander")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")


## 3. Write the abstract MDP and LTLf automaton module

In [ ]:
%%writefile abstract_mdps.py
import re
from collections import defaultdict
import numpy as np

# Importiamo il parser corretto dalla libreria ltlf2dfa
from ltlf2dfa.parser.ltlf import LTLfParser
from graphviz import Source

class LTLfAutomaton:
    """
    Wrapper per la libreria ltlf2dfa.
    Converte una formula LTLf in un DFA (formato DOT) e ne fa il parsing 
    in un grafo navigabile per l'MDP.
    """
    def __init__(self, formula_str):
        self.formula_str = formula_str
        
        # 1. Parsing della formula e generazione del DFA (in formato DOT)
        parser = LTLfParser()
        parsed_formula = parser(formula_str)
        dot_string = parsed_formula.to_dfa()
        self.dot_string = parsed_formula.to_dfa()
        
        # 2. Strutture dati dell'automa
        self.states = set()
        self.accepting_states = set()
        self.transitions = {}  # {stato_sorgente: [(condizione_booleana, stato_destinazione), ...]}
        self.initial_state = None
        
        # 3. Estrazione delle informazioni dalla stringa DOT
        self._parse_dot(dot_string)
        
        # Ordiniamo gli stati in una lista per l'MDP
        self.states = sorted(list(self.states))
        self.num_phases = len(self.states)

    def _parse_dot(self, dot_string):
        """
        Analizza la stringa DOT generata da ltlf2dfa ed estrae stati, 
        stati accettanti, stato iniziale e transizioni logiche.
        """
        # Estrazione degli stati accettanti (es. node [shape = doublecircle]; 2 3;)
        match_acc = re.search(r'node\s*\[shape\s*=\s*doublecircle\]\s*;\s*(.*?);', dot_string)
        if match_acc:
            acc_str = match_acc.group(1).replace(',', ' ')
            self.accepting_states = set(int(s) for s in acc_str.split() if s.strip().isdigit())
            
        # Estrazione delle transizioni (es. 1 -> 2 [label="wp1 & ~wp2"])
        trans_matches = re.findall(r'(\d+)\s*->\s*(\d+)\s*\[label\s*=\s*"(.*?)"\]', dot_string)
        for src_str, dst_str, guard in trans_matches:
            src = int(src_str)
            dst = int(dst_str)
            self.states.add(src)
            self.states.add(dst)
            
            if src not in self.transitions:
                self.transitions[src] = []
            self.transitions[src].append((guard, dst))
            
        # Estrazione dello stato iniziale (solitamente indicato da un arco senza etichetta da un nodo fantasma '0')
        # Es. 0 [style=invis]; 0 -> 1;
        init_match = re.search(r'(\d+)\s*->\s*(\d+)\s*;', dot_string)
        if init_match:
            self.initial_state = int(init_match.group(2))
        else:
            self.initial_state = min(self.states) if self.states else 0

    def get_initial_q(self):
        """Restituisce l'ID dello stato iniziale dell'automa."""
        return self.initial_state

    def is_goal_reached(self, current_q):
        """Verifica se lo stato attuale è uno stato accettante."""
        return current_q in self.accepting_states

    def get_next_q(self, current_q, truth_assignment):
        """
        Valuta le condizioni logiche (guardie) delle transizioni in uscita dallo 
        stato corrente e restituisce il prossimo stato dell'automa.
        """
        if current_q not in self.transitions:
            return current_q
            
        for guard, next_q in self.transitions[current_q]:
            if self._eval_guard(guard, truth_assignment):
                return next_q
                
        return current_q

    def _eval_guard(self, guard, truth_assignment):
        """
        Converte una guardia dal formato DOT (es. "wp1 & ~wp2") in Python 
        e la valuta rispetto al dizionario di verità attuale.
        """
        guard = guard.strip()
        
        # 1. Intercettiamo le costanti universali (sia formati numerici che testuali)
        if guard.lower() in ["1", "true"]: return True
        if guard.lower() in ["0", "false"]: return False
        
        # Mappiamo gli operatori logici standard in sintassi Python
        expr = guard.replace('&', ' and ').replace('|', ' or ').replace('~', ' not ').replace('!', ' not ')
        
        try:
            # 2. FIX: Usare un dizionario vuoto {} al posto di None per i builtins
            return eval(expr, {"__builtins__": {}}, truth_assignment)
        except Exception as e:
            print(f"[Errore LTLfAutomaton] Impossibile valutare la transizione '{guard}': {e}")
            return False

    def render_graph(self, filename="ltlf_automaton", directory="img"):
        """
        Renderizza e salva il DFA come immagine PNG.
        """
        try:
            src = Source(self.dot_string)
            src.render(filename=filename, directory=directory, format='png', cleanup=True)
            print(f"Grafo dell'automa salvato con successo in: {directory}/{filename}.png")
        except Exception as e:
            print(f"[Errore Graphviz] Impossibile renderizzare il grafo: {e}")


class LTLfWaypointMDP:
    """
    MDP per task sequenziali guidato da un automa LTLf.
    Lo stato astratto è 3D: (x, y, q) dove q è l'ID dello stato del DFA.
    """
    def __init__(self, waypoints_dict, ltlf_automaton, width=12, height=12, gamma=0.99, goal_reward=10000):
        self.width = width
        self.height = height
        self.gamma = gamma
        self.actions = [0, 1, 2, 3, 4, 5, 6, 7] # Include i movimenti diagonali
        
        self.waypoints_dict = waypoints_dict
        self.automaton = ltlf_automaton
        self.num_phases = self.automaton.num_phases
        
        # Generazione degli stati usando la griglia e tutti i possibili stati dell'automa
        self.states = [(x, y, q) for x in range(width) for y in range(height) for q in self.automaton.states]
        
        self.goal_reward = goal_reward
        self.v_star = defaultdict(float)
        
    def _get_truth_assignment(self, x, y):
        """
        Mappa le coordinate (x,y) attuali nelle proposizioni logiche.
        Restituisce un dizionario di verità per lo step dell'automa.
        """
        truth_assignment = {}
        for prop_name, (wp_x, wp_y) in self.waypoints_dict.items():
            truth_assignment[prop_name] = (x == wp_x and y == wp_y)
        return truth_assignment

    def get_transitions_old(self, state, action):
        x, y, q = state
        next_y = y
        reward = 0
        
        # Movimento asse Y
        if action in [0, 4, 5]:    next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:  next_y = max(y - 1, 0)
            
        # Movimento asse X
        next_x = x
        if action in [2, 4, 6]:    next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:  next_x = min(x + 1, self.width - 1)
        
        truth_assignment = self._get_truth_assignment(x, y)
        
        # Otteniamo il prossimo stato dell'automa dal modulo LTLf
        next_q = self.automaton.get_next_q(q, truth_assignment)
        next_state = (next_x, next_y, next_q)
        
        return next_state, reward

    def get_transitions(self, state, action):
        x, y, q = state
        reward = 0
        
        # 1. Calcola il movimento fisico NORMALE (come facevi prima)
        next_y = y
        if action in [0, 4, 5]:    next_y = min(y + 1, self.height - 1)
        elif action in [1, 6, 7]:  next_y = max(y - 1, 0)
            
        next_x = x
        if action in [2, 4, 6]:    next_x = max(x - 1, 0)
        elif action in [3, 5, 7]:  next_x = min(x + 1, self.width - 1)
        
        # 2. LA MODIFICA CHIAVE: Valuta le proposizioni sulle coordinate di ARRIVO
        # (Esempio generico, adatta alla tua funzione di valutazione)
        truth_assignment = self._get_truth_assignment(next_x, next_y)
        
        # 3. Aggiorna lo stato dell'automa basandoti su questa valutazione
        next_q = self.automaton.get_next_q(q, truth_assignment)

        next_state = (next_x, next_y, next_q)
        return next_state, reward
    
    def value_iteration(self, theta=0.001):
        print(f"Risoluzione MDP con Automa LTLf ({self.num_phases} Stati) tramite Value Iteration...")
        
        for s in self.states:
            if self.automaton.is_goal_reached(s[2]):
                self.v_star[s] = self.goal_reward
        
        while True:
            delta = 0
            new_v = self.v_star.copy()
            for s in self.states:
                if not self.automaton.is_goal_reached(s[2]):
                    v_actions = [self.get_transitions(s, a)[1] + self.gamma * self.v_star[self.get_transitions(s, a)[0]] for a in self.actions]
                    best_v = max(v_actions)
                    delta = max(delta, abs(best_v - self.v_star[s]))
                    new_v[s] = best_v
            self.v_star = new_v
            if delta < theta: break

    def value_iteration_old(self, theta=0.001):
        print(f"Risoluzione MDP con Automa LTLf ({self.num_phases} Stati) tramite Value Iteration...")
        
        terminal_states = set()
        
        # 1. Replichiamo la vecchia logica: ancoriamo il premio massimo allo stato 
        # fisico che *causerà* la fine, invece di aspettare lo stato accettante logico.
        for s in self.states:
            x, y, q = s
            truth_assignment = self._get_truth_assignment(x, y)
            next_q = self.automaton.get_next_q(q, truth_assignment)
            
            # Se calpestare (x, y) nello stato logico 'q' soddisfa la formula:
            if self.automaton.is_goal_reached(next_q):
                terminal_states.add(s)
                self.v_star[s] = self.goal_reward
                
            # Manteniamo a 10000 anche gli stati logici già accettanti per sicurezza
            elif self.automaton.is_goal_reached(q):
                terminal_states.add(s)
                self.v_star[s] = self.goal_reward
        
        # 2. Ciclo di update identico al precedente
        while True:
            delta = 0
            new_v = self.v_star.copy()
            for s in self.states:
                # Come nel vecchio codice (if s != self.goal_state:), 
                # saltiamo l'update per gli stati terminali per non diluire il valore
                if s not in terminal_states: 
                    v_actions = [self.get_transitions(s, a)[1] + self.gamma * self.v_star[self.get_transitions(s, a)[0]] for a in self.actions]
                    best_v = max(v_actions)
                    delta = max(delta, abs(best_v - self.v_star[s]))
                    new_v[s] = best_v
            self.v_star = new_v
            if delta < theta: break

## 4. Write the DDQN agent module

In [ ]:
%%writefile agent.py
import numpy as np
import random as ran
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class ReplayBuffer:
    def __init__(self, capacity, num_phases):
        self.capacity = capacity
        self.num_phases = num_phases
        self.buffer = []
        self.phase_indices = []
        self.phase_counts = np.zeros(num_phases, dtype=np.int64)
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        """Insert a transition and update DFA-state counts in constant time."""
        transition = (state, action, reward, next_state, done)
        phase_index = int(np.argmax(state[-self.num_phases:]))

        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
            self.phase_indices.append(phase_index)
        else:
            replaced_phase_index = self.phase_indices[self.position]
            self.phase_counts[replaced_phase_index] -= 1
            self.buffer[self.position] = transition
            self.phase_indices[self.position] = phase_index

        self.phase_counts[phase_index] += 1
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        """Sample transitions efficiently from the indexable ring buffer."""
        batch = ran.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.array, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

    def q_fraction_onehot(self, q_index, num_phases):
        """Return a DFA-state fraction using incrementally maintained counts."""
        if num_phases != self.num_phases:
            raise ValueError(f"Expected {self.num_phases} DFA states, received {num_phases}")
        if not 0 <= q_index < self.num_phases:
            raise IndexError(f"DFA state index {q_index} is out of range")
        if len(self.buffer) == 0:
            return 0.0
        return float(self.phase_counts[q_index] / len(self.buffer))

class HierarchicalDQNLearner:
    def __init__(self, env, abstract_mdp=None, max_episodes=1000, eps_decay = 0.995, gamma=0.99, policy_name="policy", use_ddqn=False, extra_state_dims=0):
        self.env = env
        self.abstract_mdp = abstract_mdp
        self.max_episodes = max_episodes
        self.gamma = gamma
        self.policy_name = policy_name
        self.use_ddqn = use_ddqn
        self.algo_name = "DDQN" if use_ddqn else "TRUE_SINGLE_DQN"
        
        self.batch_size = 64
        self.lr = 1e-3
        self.tau = 0.005 
        self.eps = 1.0
        self.eps_min = 0.01
        self.eps_decay = eps_decay
        
        # Account for dynamic one-hot phases appended to state
        state_dim = self.env.observation_space.shape[0] + extra_state_dims
        action_dim = self.env.action_space.n
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy_net = QNetwork(state_dim, action_dim).to(self.device)
        print(f"Using device:{self.device}")
        
        if self.use_ddqn:
            self.target_net = QNetwork(state_dim, action_dim).to(self.device)
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.memory = ReplayBuffer(capacity=300000, num_phases=extra_state_dims)

    def select_action(self, state):
        if ran.random() < self.eps:
            return self.env.action_space.sample()
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_tensor)
                return q_values.argmax(dim=1).item()

    def optimize_model(self):
        if len(self.memory) < self.batch_size: return
            
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)
        
        q_values = self.policy_net(states).gather(1, actions)
        
        with torch.no_grad():
            if self.use_ddqn:
                best_actions = self.policy_net(next_states).argmax(dim=1).unsqueeze(1)
                next_q_values = self.target_net(next_states).gather(1, best_actions)
            else:
                next_q_values = self.policy_net(next_states).max(1)[0].unsqueeze(1)
                
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
            
        loss = F.mse_loss(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        if self.use_ddqn:
            for target_param, policy_param in zip(self.target_net.parameters(), self.policy_net.parameters()):
                target_param.data.copy_(self.tau * policy_param.data + (1.0 - self.tau) * target_param.data)

    def _save_policy(self):
        os.makedirs("./policy", exist_ok=True)
        torch.save(self.policy_net.state_dict(), f"./policy/{self.policy_name}")


## 5. Write plotting and abstraction utilities

In [ ]:
%%writefile utils.py
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def phi_mapping_grid(obs, grid_w=12, grid_h=12):
    x, y = obs[0], obs[1]
    abstract_x = int(np.clip((x + 1) / 2 * (grid_w - 1), 0, grid_w - 1))
    abstract_y = int(np.clip(y / 1.5 * (grid_h - 1), 0, grid_h - 1))
    return abstract_x, abstract_y

def phi_mapping_sequential(obs, q, grid_w=12, grid_h=12):
    abstract_x, abstract_y = phi_mapping_grid(obs, grid_w, grid_h)
    return abstract_x, abstract_y, q

def save_sequential_heatmaps(abstract_mdp, filename_prefix="v_star"):
    """
    Generates and saves a separate heatmap for V* for each phase defined in the MDP,
    without any waypoint or goal markers (clean heatmap).
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    # Store every heatmap directly under img/heatmaps.
    output_dir = os.path.join("img", "heatmaps")
    os.makedirs(output_dir, exist_ok=True)
    filename_prefix = os.path.basename(filename_prefix)
    
    width, height = abstract_mdp.width, abstract_mdp.height
    
    # Extract global min/max for consistent colormap scaling
    all_values = np.array(list(abstract_mdp.v_star.values()))
    computed_vmin = all_values.min() if len(all_values) > 0 else 0
    computed_vmax = all_values.max() if len(all_values) > 0 else 1

    for current_q in abstract_mdp.automaton.states:
        matrix = np.zeros((height, width))
        for (x, y, q), value in abstract_mdp.v_star.items():
            if q == current_q and 0 <= x < width and 0 <= y < height:
                matrix[y, x] = value
                
        plt.figure(figsize=(9, 8))
        im = plt.imshow(matrix, cmap='viridis', origin='lower', vmin=computed_vmin, vmax=computed_vmax)
        
        for y in range(height):
            for x in range(width):
                val = matrix[y, x]
                if val > 0.0: 
                    text_color = 'white' if val < (computed_vmax / 2) else 'black'
                    plt.text(x, y, f"{val:.1f}", ha='center', va='center', color=text_color, fontsize=7)
                    
        plt.colorbar(im, fraction=0.046, pad=0.04, label="Potential Value (V*)")
        
        is_goal_state = abstract_mdp.automaton.is_goal_reached(current_q)
        phase_label = "Goal Reached" if is_goal_state else "Seeking Targets"
        plt.title(f"Potential Map (V*) - DFA State q={current_q} ({phase_label})", fontsize=14, fontweight='bold')
        
        ax = plt.gca()
        ax.set_xticks(np.arange(-.5, width, 1), minor=True)
        ax.set_yticks(np.arange(-.5, height, 1), minor=True)
        ax.grid(which='minor', color='w', linestyle='-', linewidth=1, alpha=0.4)
        
        # Nessun plot per i waypoint o il goal
            
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{filename_prefix}_q{current_q}.png"), dpi=150, bbox_inches='tight')
        plt.close()
        print(f" -> Generated V* Heatmap for DFA State q={current_q}")

def plot_comparison_curves(baseline_rewards, shaping_rewards, epsilon_history=None, window_size=100, filename="img/baseline_vs_shaping.png", title="Learning Curve Comparison", baseline_label="Baseline", shaping_label="Shaping"):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    baseline_ma = pd.Series(baseline_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    shaping_ma = pd.Series(shaping_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    x_axis = np.arange(len(baseline_rewards))
    ax1.plot(x_axis, baseline_ma, color='black', linestyle='-', linewidth=2, label=baseline_label)
    ax1.plot(x_axis, shaping_ma, color='blue', linestyle='-', linewidth=2.5, label=shaping_label)
    ax1.set_title(title, fontsize=15, fontweight='bold')
    ax1.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    if epsilon_history:
        ax2 = ax1.twinx()
        ax2.plot(x_axis, epsilon_history, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')
        ax2.set_ylabel("Exploration Rate (ε)", color='orange', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='orange')
        ax2.set_ylim(0, 1.05)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=11)
    else:
        ax1.legend(loc="lower right", fontsize=11)

    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Comparison plot successfully saved to: {filename}")
    plt.close(fig)

def plot_mean_std_curves(reward_histories_single=None, reward_histories_multi=None, window_size=100, title="Mean Performance with Variance", filename="img/mean_std_plot.png"):
    """
    Plots the mean and standard deviation of reward histories for one or two sets of runs.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    def plot_single_curve(reward_histories, label, color):
        if not reward_histories:
            return
        
        # Ensure all histories have the same length by padding with NaNs if necessary
        max_len = max(len(h) for h in reward_histories)
        padded_histories = [np.pad(h, (0, max_len - len(h)), 'constant', constant_values=np.nan) for h in reward_histories]
        
        rewards_df = pd.DataFrame(padded_histories).T
        mean_rewards = rewards_df.mean(axis=1)
        std_rewards = rewards_df.std(axis=1)

        # Apply moving average
        mean_ma = mean_rewards.rolling(window=window_size, min_periods=1, center=True).mean()
        std_ma = std_rewards.rolling(window=window_size, min_periods=1, center=True).mean()

        x_axis = np.arange(len(mean_ma))
        ax.plot(x_axis, mean_ma, label=f"Mean {label}", color=color, linewidth=2.5)
        ax.fill_between(x_axis, mean_ma - std_ma, mean_ma + std_ma, color=color, alpha=0.2, label=f"Std Dev {label}")

    if reward_histories_single:
        plot_single_curve(reward_histories_single, "Single Epsilon", "black")

    if reward_histories_multi:
        plot_single_curve(reward_histories_multi, "Multi Epsilon", "blue")

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel(f"Episode # (Moving Average Window = {window_size})", fontsize=12)
    ax.set_ylabel("Mean Episode Reward", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc="lower right", fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    print(f"\n>>> Mean/Std plot successfully saved to: {filename}")
    plt.close(fig)

def plot_buffer_fractions(buffer_histories, window_size=100, filename="img/buffer_fractions.png", state_labels=None):
    """
    Plots the replay buffer composition for N phases dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    x_axis = np.arange(len(buffer_histories[0]))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(buffer_histories)))
    for idx, history in enumerate(buffer_histories):
        ma = pd.Series(history).rolling(window=window_size, min_periods=1, center=True).mean()
        state_label = state_labels[idx] if state_labels is not None else idx
        ax.plot(x_axis, ma, color=colors[idx], linewidth=2.5, label=f'DFA state q={state_label}')
    
    ax.set_title(f"Replay Buffer Composition (MA Window = {window_size})", fontsize=14, fontweight='bold')
    ax.set_ylabel("Fraction in Buffer", fontsize=12)
    ax.set_ylim(0, 1.05)
    
    ideal_balance = 1.0 / len(buffer_histories)
    ax.axhline(y=ideal_balance, color='gray', linestyle=':', alpha=0.7, label=f'Ideal Balance ({ideal_balance:.0%})')
    
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(buffer_histories)+1, fontsize=11)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)

def plot_shaping_reward_breakdown(true_rewards, total_rewards, eps_histories, window_size=100, filename="img/shaping_reward_breakdown.png"):
    """
    Plots the moving average of rewards (True vs Total) and overlays the N-phase Epsilon decay dynamically.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Moving Average Calculation
    if len(true_rewards) >= window_size:
        true_ma = pd.Series(true_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
        total_ma = pd.Series(total_rewards).rolling(window=window_size, min_periods=1, center=True).mean()
    else:
        true_ma = true_rewards
        total_ma = total_rewards
        
    x_axis = np.arange(len(true_rewards))
        
    # Plot Rewards (Left Y-Axis)
    ax1.plot(x_axis, true_ma, color='green', linestyle='-', linewidth=2, label='Synthetic Goal Reward')
    ax1.plot(x_axis, total_ma, color='purple', linestyle='-', linewidth=2.5, label='Learning Reward (Goal + Shaping)')
    
    ax1.set_title(f"Shaping Agent Reward Analysis (MA Window = {window_size})", fontsize=15, fontweight='bold')
    ax1.set_xlabel("Episode #", fontsize=12)
    ax1.set_ylabel("Episode Reward", fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Plot Epsilon Decays (Right Y-Axis)
    ax2 = ax1.twinx()
    
    # Check if eps_histories is a list of lists/arrays (multi-epsilon case)
    is_multi_eps = any(isinstance(i, (list, np.ndarray)) for i in eps_histories)

    if is_multi_eps:
        num_phases = len(eps_histories)
        colors = plt.cm.plasma(np.linspace(0, 0.8, num_phases))
        for idx in range(num_phases):
            label = "Goal" if idx == num_phases - 1 else f"WP {idx + 1}"
            ax2.plot(x_axis, eps_histories[idx], color=colors[idx], linestyle='--', linewidth=2, alpha=0.8, label=f'ε Decay (q={idx}: {label})')
    else: # Single epsilon history
        ax2.plot(x_axis, eps_histories, color='orange', linestyle='--', linewidth=1.8, label='Epsilon Decay')

    # Align the zero of both y-axes for better visual comparison
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = -0.05, 1.05 # Epsilon range is fixed
    
    # Align y-axes so that the zero points match.
    if y1_min < 0 < y1_max:
        # Calculate the proportional position of zero on the reward axis
        zero_ratio = -y1_min / (y1_max - y1_min)
        # Set the epsilon axis limits so its zero is at the same ratio
        new_y2_min = -zero_ratio * y2_max / (1 - zero_ratio)
        ax2.set_ylim(new_y2_min, y2_max)
    else:
        ax2.set_ylim(y2_min, y2_max)

    ax2.set_ylabel("Exploration Rate (ε)", color='black', fontsize=12)

    # Combine Legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Dynamically calculate legend columns based on number of items to keep it compact
    legend_cols = max(2, (len(labels1) + len(labels2)) // 2)
    
    ax1.legend(
        lines1 + lines2, labels1 + labels2, 
        loc="upper center", bbox_to_anchor=(0.5, -0.15), 
        ncol=legend_cols, fontsize=11, framealpha=1.0
    )
    
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)


## 6. Write the LTLf task configuration

In [ ]:
%%writefile trajectory.json
{
	"formula": "F(wp1 & X(F(goal)))",
	"grid_w": 12,
	"grid_h": 12,
	"goal_reward": 10000,
	"waypoints_dict": {
		"wp1": [1,8],
		"goal": [8,8]
	}
}

## 7. Write the training program

In [ ]:
%%writefile trainer.py
# ==============================
# Standard library imports
# ==============================

import argparse
import json
import os
from collections import Counter

# ==============================
# External and project imports
# ==============================

import gymnasium as gym
import numpy as np

from abstract_mdps import LTLfAutomaton, LTLfWaypointMDP
from agent import HierarchicalDQNLearner
from utils import phi_mapping_sequential, plot_buffer_fractions, plot_shaping_reward_breakdown, save_sequential_heatmaps


# ==============================
# Data and state helpers
# ==============================

def save_training_data(filename, **kwargs):
    """Convert training metrics to arrays and save them in a compressed NPZ file."""
    # Preserve numeric dtypes and rectangular shapes for direct plotting.
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    np_data = {key: np.asarray(value) for key, value in kwargs.items()}
    if any(array.dtype == object for array in np_data.values()):
        raise ValueError("Training metrics must be rectangular numeric arrays")
    np.savez_compressed(filename, **np_data)
    print(f"\nTraining data saved to: {filename}")


def _abstract_position(observation):
    """Map a raw environment observation to its abstract spatial coordinates."""
    x, y, _ = phi_mapping_sequential(observation, 0)
    return x, y


def _augment_state(observation, q, state_to_index):
    """Append a one-hot encoding of the current DFA state to an observation."""
    one_hot = np.zeros(len(state_to_index), dtype=np.float32)
    one_hot[state_to_index[q]] = 1.0
    return np.concatenate((observation, one_hot)).astype(np.float32)


def _evaluate_initial_automaton_state(observation, abstract_mdp):
    """Consume the initial observation from the DFA pre-trace state and return the first active state."""
    initial_x, initial_y = _abstract_position(observation)
    initial_truth_assignment = abstract_mdp._get_truth_assignment(initial_x, initial_y)
    pre_trace_q = abstract_mdp.automaton.get_initial_q()
    return abstract_mdp.automaton.get_next_q(pre_trace_q, initial_truth_assignment)


def _format_counter(counter):
    """Convert a DFA transition counter into a compact human-readable string."""
    if not counter:
        return "none"
    return ", ".join(f"{source}->{destination}: {count}" for (source, destination), count in sorted(counter.items()))


# ==============================
# Logging and checkpoint helpers
# ==============================

def _write_log(message, log_handle=None):
    """Print a message and optionally append it to the active log file."""
    print(message)
    if log_handle:
        log_handle.write(message)
        log_handle.flush()


def _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states):
    """Write the configuration and DFA metadata at the beginning of a training run."""
    if not log_handle:
        return
    automaton = abstract_mdp.automaton
    header = (
        "\n=== NEW RUN ===\n"
        f"episodes={episodes}, shaping={use_shaping}, K={K}, goal_reward={goal_reward}, gamma={abstract_mdp.gamma}\n"
        f"formula={automaton.formula_str}\n"
        f"waypoints={abstract_mdp.waypoints_dict}\n"
        f"dfa_states={automaton_states}, pre_trace={automaton.get_initial_q()}, accepting={sorted(automaton.accepting_states)}\n"
    )
    log_handle.write(header)
    log_handle.flush()


def _should_log(episode, episodes, log_interval):
    """Return whether the current episode requires a periodic training report."""
    return episode == 0 or episode + 1 == episodes or (episode + 1) % log_interval == 0


def _build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters):
    """Build a report containing recent metrics and cumulative DFA counters."""
    window = min(log_interval, episode + 1)
    recent_slice = slice(-window, None)
    recent_transitions = Counter()
    for transitions in histories["transition_counters"][-window:]:
        recent_transitions.update(transitions)

    recent_state_visits = np.asarray(histories["state_visits"], dtype=np.int64)[:, -window:].sum(axis=1)
    recent_state_entries = np.asarray(histories["state_entries"], dtype=np.int64)[:, -window:].sum(axis=1)
    buffer_details = ", ".join(f"{q}: {agent.memory.q_fraction_onehot(index, len(automaton_states)):.1%}" for index, q in enumerate(automaton_states))
    recent_visits_details = ", ".join(f"{q}: {recent_state_visits[index]}" for index, q in enumerate(automaton_states))
    recent_entries_details = ", ".join(f"{q}: {recent_state_entries[index]}" for index, q in enumerate(automaton_states))
    cumulative_visits_details = ", ".join(f"{q}: {cumulative_counters['state_visits'][q]}" for q in automaton_states)
    cumulative_entries_details = ", ".join(f"{q}: {cumulative_counters['state_entries'][q]}" for q in automaton_states)

    return (
        "\n"
        f"[Episode {episode + 1}/{episodes} | last {window}]\n"
        f"success rate                : {np.mean(histories['successes'][recent_slice]):.1%} (cumulative {np.mean(histories['successes']):.1%})\n"
        f"synthetic task reward       : {np.mean(histories['task_rewards'][recent_slice]):.3f}\n"
        f"shaping reward              : {np.mean(histories['shaping_rewards'][recent_slice]):.3f}\n"
        f"learning reward             : {np.mean(histories['learning_rewards'][recent_slice]):.3f}\n"
        f"episode length              : {np.mean(histories['episode_lengths'][recent_slice]):.1f}\n"
        f"abstract changes / episode  : {np.mean(histories['abstract_changes'][recent_slice]):.1f}\n"
        f"DFA transitions / episode   : {np.mean(histories['dfa_transitions'][recent_slice]):.2f}\n"
        f"DFA transitions in window   : {_format_counter(recent_transitions)}\n"
        f"epsilon (next episode)       : {histories['epsilons'][-1]:.5f}\n"
        f"replay buffer                : {len(agent.memory)} samples [{buffer_details}]\n"
        f"DFA state visits in window   : {recent_visits_details}\n"
        f"DFA state visits cumulative  : {cumulative_visits_details}\n"
        f"DFA state entries in window  : {recent_entries_details}\n"
        f"DFA state entries cumulative : {cumulative_entries_details}\n"
        f"transitions cumulative       : {_format_counter(cumulative_counters['transitions'])}\n"
        f"accepted directly from s0    : {cumulative_counters['initial_acceptances']}\n"
        f"Gym endings cumulative       : terminated={cumulative_counters['env_terminated']}, truncated={cumulative_counters['env_truncated']}\n"
    )


def _save_named_policy(agent, policy_name):
    """Save the current policy using a stable descriptive filename."""
    agent.policy_name = policy_name
    agent._save_policy()


def _monitoring_average(values, episode, log_interval):
    """Return the mean over the active monitoring window."""
    window = min(log_interval, episode + 1)
    return float(np.mean(values[-window:]))


def _validate_training_setup(automaton, state_to_index, episodes, log_interval):
    """Validate DFA consistency and the numeric parameters required by training."""
    if automaton.get_initial_q() not in state_to_index:
        raise ValueError("The DFA initial state is missing from automaton.states")
    if not automaton.accepting_states.issubset(state_to_index):
        raise ValueError("At least one accepting DFA state is missing from automaton.states")
    if episodes <= 0:
        raise ValueError("episodes must be greater than zero")
    if log_interval <= 0:
        raise ValueError("log_interval must be greater than zero")


def _build_training_results(histories, initial_acceptance_history, buffer_histories, automaton_states, best_mean_reward, best_policy_episode):
    """Select and name the numeric histories returned by the training loop."""
    return {
        "task_rewards": histories["task_rewards"],
        "learning_rewards": histories["learning_rewards"],
        "shaping_rewards": histories["shaping_rewards"],
        "epsilon_history": histories["epsilons"],
        "buffer_histories": buffer_histories,
        "state_visit_histories": histories["state_visits"],
        "state_entry_histories": histories["state_entries"],
        "successes": histories["successes"],
        "initial_acceptances": initial_acceptance_history,
        "episode_lengths": histories["episode_lengths"],
        "abstract_changes": histories["abstract_changes"],
        "dfa_transitions": histories["dfa_transitions"],
        "automaton_states": automaton_states,
        "best_mean_learning_reward": best_mean_reward,
        "best_policy_episode": best_policy_episode,
    }


# ==============================
# Training loop
# ==============================

def run_sequential_training(env, agent, abstract_mdp, episodes, goal_reward=10000, save_policy=True, use_shaping=True, K=1.0, log_file=None, log_interval=100):
    """
    Train the DDQN agent with the LTLf automaton and one global epsilon.

    The Gym reward is deliberately discarded. The learning reward is the
    synthetic goal reward plus potential-based shaping. Shaping is evaluated
    only when the complete abstract state (x, y, q) changes.
    """
    # Build a stable mapping between DFA states and neural-network features.
    automaton = abstract_mdp.automaton
    automaton_states = list(automaton.states)
    state_to_index = {q: index for index, q in enumerate(automaton_states)}
    num_states = len(automaton_states)

    # Fail early if the DFA or training parameters are inconsistent.
    _validate_training_setup(automaton, state_to_index, episodes, log_interval)

    # Store episode-level metrics for plots and post-processing.
    task_reward_history = []
    learning_reward_history = []
    shaping_reward_history = []
    epsilon_history = []
    episode_length_history = []
    success_history = []
    initial_acceptance_history = []
    abstract_change_history = []
    dfa_transition_history = []
    transition_counter_history = []
    buffer_histories = [[] for _ in automaton_states]
    state_visit_histories = [[] for _ in automaton_states]
    state_entry_histories = [[] for _ in automaton_states]
    histories = {
        "task_rewards": task_reward_history,
        "learning_rewards": learning_reward_history,
        "shaping_rewards": shaping_reward_history,
        "epsilons": epsilon_history,
        "episode_lengths": episode_length_history,
        "successes": success_history,
        "abstract_changes": abstract_change_history,
        "dfa_transitions": dfa_transition_history,
        "transition_counters": transition_counter_history,
        "state_visits": state_visit_histories,
        "state_entries": state_entry_histories,
    }

    # Keep cumulative counters for diagnostics shown during training.
    cumulative_state_visits = Counter()
    cumulative_state_entries = Counter()
    cumulative_transitions = Counter()
    cumulative_env_terminated = 0
    cumulative_env_truncated = 0
    cumulative_initial_acceptances = 0
    best_mean_reward = -np.inf
    best_policy_episode = 0

    # Open one append-only log file for the complete run.
    log_handle = open(log_file, "a", encoding="utf-8") if log_file else None
    _write_run_header(log_handle, episodes, use_shaping, K, goal_reward, abstract_mdp, automaton_states)

    try:
        for episode in range(episodes):
            # Reset the environment and consume s0 before selecting the first action.
            raw_state, _ = env.reset()
            q = _evaluate_initial_automaton_state(raw_state, abstract_mdp)
            if q not in state_to_index:
                raise RuntimeError(f"DFA returned unknown initial state {q!r} after evaluating s0")
            augmented_state = _augment_state(raw_state, q, state_to_index)

            # Reset counters local to the current episode.
            succeeded = automaton.is_goal_reached(q)
            done = succeeded
            episode_steps = 0
            episode_task_reward = float(goal_reward) if succeeded else 0.0
            episode_shaping_reward = 0.0
            episode_abstract_changes = 0
            episode_dfa_transitions = 0
            episode_state_visits = [0] * num_states
            episode_state_visits[state_to_index[q]] = 1
            # Count s0 as an entry from the virtual pre-trace state.
            episode_state_entries = [0] * num_states
            episode_state_entries[state_to_index[q]] = 1
            episode_transitions = Counter()
            if succeeded:
                cumulative_initial_acceptances += 1
            cumulative_state_visits[q] += 1
            cumulative_state_entries[q] += 1

            while not done:
                # Select an action using the single global epsilon.
                agent.eps = epsilon_history[-1] if epsilon_history else agent.eps
                action = agent.select_action(augmented_state)

                # The environment reward is intentionally not part of training.
                next_raw_state, _ignored_env_reward, env_terminated, env_truncated, _ = env.step(action)

                # Map the transition to abstract spatial states.
                x, y = _abstract_position(raw_state)
                next_x, next_y = _abstract_position(next_raw_state)
                abstract_state = (x, y, q)

                # Advance the DFA using propositions true in the arrival state.
                truth_assignment = abstract_mdp._get_truth_assignment(next_x, next_y)
                next_q = automaton.get_next_q(q, truth_assignment)
                if next_q not in state_to_index:
                    raise RuntimeError(f"DFA returned unknown state {next_q!r} from state {q!r}")

                # Count every arrival in a DFA state, including self-transitions.
                episode_state_visits[state_to_index[next_q]] += 1
                cumulative_state_visits[next_q] += 1

                # Track physical abstraction changes separately from DFA changes.
                abstract_next_state = (next_x, next_y, next_q)
                abstract_changed = abstract_state != abstract_next_state
                dfa_changed = next_q != q

                if abstract_changed:
                    episode_abstract_changes += 1
                if dfa_changed:
                    transition = (q, next_q)
                    episode_dfa_transitions += 1
                    episode_state_entries[state_to_index[next_q]] += 1
                    episode_transitions[transition] += 1
                    cumulative_state_entries[next_q] += 1
                    cumulative_transitions[transition] += 1

                # Assign the synthetic task reward only on DFA acceptance.
                synthetic_goal_reward = 0.0
                if automaton.is_goal_reached(next_q):
                    synthetic_goal_reward = float(goal_reward)
                    succeeded = True

                # A successful DFA transition ends the episode even if Gym would continue.
                done = env_terminated or env_truncated or succeeded
                next_augmented_state = _augment_state(next_raw_state, next_q, state_to_index)

                # Evaluate shaping only when the complete abstract state changes.
                shaping_signal = 0.0
                if use_shaping and abstract_changed:
                    phi_state = abstract_mdp.v_star.get(abstract_state, 0.0)
                    phi_next_state = abstract_mdp.v_star.get(abstract_next_state, 0.0)
                    shaping_signal = K * (abstract_mdp.gamma * phi_next_state - phi_state)

                # Store the transition and perform one DDQN optimization step.
                learning_reward = synthetic_goal_reward + shaping_signal
                agent.memory.push(augmented_state, action, learning_reward, next_augmented_state, done)
                agent.optimize_model()

                # Update the episode totals and move to the next state.
                episode_steps += 1
                episode_task_reward += synthetic_goal_reward
                episode_shaping_reward += shaping_signal
                raw_state = next_raw_state
                augmented_state = next_augmented_state
                q = next_q

                # Count Gym endings for diagnostics without using its reward.
                if env_terminated:
                    cumulative_env_terminated += 1
                if env_truncated:
                    cumulative_env_truncated += 1

            # Decay the single epsilon once at the end of the episode.
            next_epsilon = max(agent.eps_min, agent.eps * agent.eps_decay)
            agent.eps = next_epsilon

            # Save the metrics collected for this episode.
            episode_learning_reward = episode_task_reward + episode_shaping_reward
            task_reward_history.append(episode_task_reward)
            shaping_reward_history.append(episode_shaping_reward)
            learning_reward_history.append(episode_learning_reward)
            epsilon_history.append(next_epsilon)
            episode_length_history.append(episode_steps)
            success_history.append(int(succeeded))
            initial_acceptance_history.append(int(episode_steps == 0 and succeeded))
            abstract_change_history.append(episode_abstract_changes)
            dfa_transition_history.append(episode_dfa_transitions)
            transition_counter_history.append(episode_transitions)

            # Record replay-buffer composition, state visits, and entries from other states.
            for index in range(num_states):
                buffer_histories[index].append(agent.memory.q_fraction_onehot(index, num_states))
                state_visit_histories[index].append(episode_state_visits[index])
                state_entry_histories[index].append(episode_state_entries[index])

            # Print recent and cumulative diagnostics at the requested interval.
            if _should_log(episode, episodes, log_interval):
                cumulative_counters = {"state_visits": cumulative_state_visits, "state_entries": cumulative_state_entries, "transitions": cumulative_transitions, "initial_acceptances": cumulative_initial_acceptances, "env_terminated": cumulative_env_terminated, "env_truncated": cumulative_env_truncated}
                _write_log(_build_training_log(episode, episodes, log_interval, automaton_states, agent, histories, cumulative_counters), log_handle)

                # Replace the best policy when the monitored mean reward improves.
                monitored_mean_reward = _monitoring_average(learning_reward_history, episode, log_interval)
                if monitored_mean_reward > best_mean_reward:
                    best_mean_reward = monitored_mean_reward
                    best_policy_episode = episode + 1
                    if save_policy:
                        _save_named_policy(agent, "best_policy.pth")
                        _write_log(f"Best policy updated at episode {best_policy_episode}: mean learning reward={best_mean_reward:.3f}\n", log_handle)

        # Save the final policy independently from its monitored performance.
        if save_policy:
            _save_named_policy(agent, "last_policy.pth")
            _write_log(f"Last policy saved after episode {episodes}. Best policy: episode {best_policy_episode}, mean learning reward={best_mean_reward:.3f}\n", log_handle)
    finally:
        # Always close the log, including when training raises an exception.
        if log_handle:
            log_handle.close()

    # Return named histories to avoid ambiguous tuple positions.
    return _build_training_results(histories, initial_acceptance_history, buffer_histories, automaton_states, best_mean_reward, best_policy_episode)


# ==============================
# Experiment setup and outputs
# ==============================

def main(args):
    """Configure the experiment, run or load training, and generate diagnostic plots."""
    # Prepare output directories shared by training and post-processing.
    data_dir = "results"
    image_dir = "img"
    log_dir = "logs"
    for directory in (data_dir, image_dir, log_dir):
        os.makedirs(directory, exist_ok=True)
    plot_dir = data_dir if args.post_process else image_dir

    # Load the temporal task and optional training parameters.
    with open(args.config, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)

    formula = config.get("formula", "F(goal)")
    waypoints = {name: tuple(coordinates) for name, coordinates in config.get("waypoints_dict", {"goal": [5, 0]}).items()}
    gamma = float(config.get("gamma", 0.99))
    goal_reward = float(config.get("goal_reward", 10000))

    # Build the DFA once for both training and post-processing.
    automaton = LTLfAutomaton(formula)
    print(
        "=== LTLf TRAINING (single epsilon) ===\n"
        f"Formula: {formula}\n"
        f"Waypoints: {waypoints}\n"
        f"DFA: states={automaton.states}, pre-trace={automaton.initial_state}, "
        f"accepting={sorted(automaton.accepting_states)}\n"
        "Gym reward is ignored by design."
    )

    if not args.post_process:
        # Create the environment and abstract MDP used to compute the potential.
        automaton.render_graph()
        env = gym.make("LunarLander-v3", continuous=False)
        try:
            abstract_mdp = LTLfWaypointMDP(waypoints_dict=waypoints, ltlf_automaton=automaton, width=int(config.get("grid_w", 12)), height=int(config.get("grid_h", 12)), gamma=gamma, goal_reward=goal_reward)
            abstract_mdp.value_iteration()
            save_sequential_heatmaps(abstract_mdp, filename_prefix="single_epsilon_exp")

            # Initialize one DDQN agent with a single exploration schedule.
            agent = HierarchicalDQNLearner(env=env, max_episodes=args.episodes, eps_decay=args.eps_decay, use_ddqn=True, extra_state_dims=len(automaton.states))

            # Run training and persist all collected metrics.
            metrics = run_sequential_training(env=env, agent=agent, abstract_mdp=abstract_mdp, episodes=args.episodes, goal_reward=goal_reward, use_shaping=not args.no_shaping, K=args.shaping_scale, log_file=f"{log_dir}/single_epsilon_training.log", log_interval=args.log_interval)
            save_training_data(f"{data_dir}/single_epsilon_data.npz", **metrics)
        finally:
            # Release environment resources even if training fails.
            env.close()

    # Load saved metrics and generate the final diagnostic plots.
    data = np.load(f"{data_dir}/single_epsilon_data.npz", allow_pickle=False)
    plot_buffer_fractions(data["buffer_histories"], filename=f"{plot_dir}/buffer_fractions_single_epsilon.png", window_size=args.plot_window, state_labels=data["automaton_states"])
    plot_shaping_reward_breakdown(data["task_rewards"], data["learning_rewards"], data["epsilon_history"], window_size=args.plot_window, filename=f"{plot_dir}/reward_breakdown_single_epsilon.png")
    print("\nFinished.")


# ==============================
# Command-line entry point
# ==============================

if __name__ == "__main__":
    # Expose the main training and post-processing options.
    parser = argparse.ArgumentParser(description="LTLf DDQN training with one global epsilon.")
    parser.add_argument("--episodes", type=int, default=1000)
    parser.add_argument("--config", default="trajectory.json")
    parser.add_argument("--eps-decay", type=float, default=0.9996)
    parser.add_argument("--shaping-scale", type=float, default=1.0)
    parser.add_argument("--log-interval", type=int, default=100)
    parser.add_argument("--plot-window", type=int, default=500)
    parser.add_argument("--no-shaping", action="store_true")
    parser.add_argument("--post-process", action="store_true")
    main(parser.parse_args())


## 8. Configure the experiment

The Gym reward remains ignored by the trainer. The best policy is selected using the mean learning reward over the logging window.

In [ ]:
EPISODES = 1000
EPSILON_DECAY = 0.9996
SHAPING_SCALE = 1.0
LOG_INTERVAL = 100
PLOT_WINDOW = 500
DISABLE_SHAPING = False

print(f"Episodes: {EPISODES}")
print(f"Epsilon decay: {EPSILON_DECAY}")
print(f"Shaping scale: {SHAPING_SCALE}")
print(f"Log interval: {LOG_INTERVAL}")


## 9. Run training

In [ ]:
import os
import subprocess

command = ["python", "trainer.py", "--episodes", str(EPISODES), "--config", "trajectory.json", "--eps-decay", str(EPSILON_DECAY), "--shaping-scale", str(SHAPING_SCALE), "--log-interval", str(LOG_INTERVAL), "--plot-window", str(PLOT_WINDOW)]
if DISABLE_SHAPING:
    command.append("--no-shaping")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
print("Running:", " ".join(command))
subprocess.run(command, cwd=WORK_DIR, env=environment, check=True)


## 10. Inspect saved metrics, logs, and policies

In [ ]:
import numpy as np

data_path = WORK_DIR / "results" / "single_epsilon_data.npz"
metrics = np.load(data_path, allow_pickle=False)

print("Saved metrics:")
for key in metrics.files:
    value = metrics[key]
    print(f"- {key}: shape={value.shape}, dtype={value.dtype}")

print(f"\nBest policy episode: {int(metrics['best_policy_episode'])}")
print(f"Best mean learning reward: {float(metrics['best_mean_learning_reward']):.3f}")
print(f"Overall success rate: {metrics['successes'].mean():.2%}")

log_path = WORK_DIR / "logs" / "single_epsilon_training.log"
if log_path.exists():
    print("\nLast log lines:\n")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-30:]))

print("\nSaved policies:")
for policy_path in sorted((WORK_DIR / "policy").glob("*.pth")):
    print(f"- {policy_path.name}")


## 11. Package outputs for download

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/lunar_lander_ltlf_outputs.zip")
output_directories = ("results", "img", "logs", "policy")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in output_directories:
        output_directory = WORK_DIR / directory_name
        if output_directory.exists():
            for output_path in sorted(output_directory.rglob("*")):
                if output_path.is_file():
                    archive.write(output_path, output_path.relative_to(WORK_DIR))
    configuration_path = WORK_DIR / "trajectory.json"
    if configuration_path.is_file():
        archive.write(configuration_path, configuration_path.name)

print(f"Output archive ready: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")
